### 01 - Instalação (Bibliotecas)

In [ ]:
!pip install torch torchvision torchaudio
!pip install openai-whisper

### 02 - Recursos (Importações)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

### 03 - Funções

In [ ]:
def tokenize(sentence):
  return sentence.lower().split()

def encode(sentence, vocabulary, default):
  return [vocabulary.get(token, default) for token in tokenize(sentence)]

def pad_sequence(sequence, max_length):
  return [0] * (max_length - len(sequence)) + sequence

def predict(sentence, vocabulary, default, model, max_length):
  model.eval()

  sequence = encode(sentence, vocabulary, default)
  sequence = pad_sequence(sequence, max_length)

  sequence = torch.tensor([sequence])

  with torch.no_grad():
    output = model(sequence).item()

  return f"Positivo ({output:.15f})" if output > 0.5 else f"Negativo ({output:.15f})"

### 04 - Classe

In [ ]:
class RNN_Sentiment(nn.Module):
  def __init__(self, vocabulary_size, embedding_dim, hidden_dim):
    super(RNN_Sentiment, self).__init__()

    self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=0)

    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

    self.fc = nn.Linear(hidden_dim, 1)

    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)

    _, hidden = self.rnn(x)

    output = self.fc(hidden[-1])

    return self.sigmoid(output)

### 05 - Implementação

In [ ]:
sentences = [
  ("Eu amo esse filme.", 1),
  ("Esse filme é ótimo.", 1),
  ("Experiência incrível.", 1),
  ("Eu odeio esse filme!", 0),
  ("Esse filme é terrível!", 0),
  ("Experiência horrível!", 0),
]

UNK_TOKEN = 1

vocabulary = { "<UNK>": UNK_TOKEN }

for sentence, _ in sentences:
 for token in tokenize(sentence):
  if (token not in vocabulary):
   vocabulary[token] = len(vocabulary) + 1

vocabulary_size = len(vocabulary) + 1

encoded_data = [(encode(sentence, vocabulary, UNK_TOKEN), label) for sentence, label in sentences]

max_length = max(len(sequence) for sequence, _ in encoded_data)

x = torch.tensor([pad_sequence(sequence, max_length) for sequence, _ in encoded_data])

y = torch.tensor([label for _, label in encoded_data], dtype=torch.float32)

print("Tensores:\n")

print(x)

embedding_dim = 10

hidden_dim = 16

model = RNN_Sentiment(
 vocabulary_size,
 embedding_dim,
 hidden_dim
)

### 06 - Treinamento

In [ ]:
criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(0, epochs):
  model.train()

  outputs = model(x).squeeze()

  loss = criterion(outputs, y)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if ((epoch + 1) % 10 == 0):
    print(
      f"| Época: {(epoch + 1):10}",
      f"| Perda: {loss.item():10.5f} |"
    )

### 07 - Aplicação

In [ ]:
print("Testes:")

print(
  f"\n| Frase: {'Eu amo esse filme.':30}",
  f"| Resultado: {predict('Eu amo esse filme.', vocabulary, UNK_TOKEN, model, max_length)} |"
)

print(
  f"\n| Frase: {'Esse filme é terrível!':30}",
  f"| Resultado: {predict('Esse filme é terrível!', vocabulary, UNK_TOKEN, model, max_length)} |"
)